<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Банк</h2> 

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.
Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

// ==========================================
// 1. ИНТЕРФЕЙСЫ
// ==========================================
public interface IRefundable { bool Refund(decimal amount); }
public interface ISecurePayment { bool IsSecure { get; } string GetSecurityInfo(); }
public interface ILoggable { void Log(string message); }
public interface IAuditable { void PerformAudit(); string GetAuditTrail(); }

// ==========================================
// [NEW] ДЕЛЕГАТЫ И АРГУМЕНТЫ СОБЫТИЙ
// ==========================================
public delegate void PaymentNotificationHandler(PaymentMethod sender, PaymentEventArgs args);

public class PaymentEventArgs : EventArgs
{
    public decimal Amount { get; }
    public bool Success { get; }
    public string Message { get; }
    public DateTime Timestamp { get; }

    public PaymentEventArgs(decimal amount, bool success, string message)
    {
        Amount = amount; Success = success; Message = message; Timestamp = DateTime.Now;
    }
}

// ==========================================
// 2. БАЗОВЫЙ КЛАСС: PaymentMethod
// ==========================================
public class PaymentMethod : IRefundable, IAuditable
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private string _currency;
    private bool _isActive;
    private string _description;
    private DateTime _createdDate;
    private decimal _taxRate;
    private bool _isTaxIncluded;
    private string _receiptNumber;
    private string _departmentCode;

    // [NEW] 4 атрибута
    private int _transactionCount;
    private decimal _totalProcessedAmount;
    private DateTime _lastTransactionDate;
    private bool _isFrozen;

    private readonly List<string> _transactionHistory = new List<string>();

    // [NEW] Делегат-поле
    public Func<decimal, bool> CustomAmountValidator { get; set; }

    // [NEW] События
    public event PaymentNotificationHandler PaymentProcessed;
    public event PaymentNotificationHandler PaymentFailed;
    public event EventHandler StatusChanged;

    // Свойства
    public int PaymentMethodId { get => _paymentMethodId; set => _paymentMethodId = value > 0 ? value : throw new ArgumentException("ID must be positive"); }
    public string MethodName { get => _methodName; set => _methodName = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Name cannot be empty"); }
    public decimal MinAmount { get => _minAmount; set => _minAmount = value >= 0 ? value : throw new ArgumentException("Min amount cannot be negative"); }
    public string Currency { get => _currency; set => _currency = (!string.IsNullOrWhiteSpace(value) && value.Length == 3) ? value.ToUpper() : throw new ArgumentException("Currency must be 3 letters"); }
    public bool IsActive { get => _isActive; set => _isActive = value; }
    public string Description { get => _description; set => _description = value ?? string.Empty; }
    public DateTime CreatedDate { get => _createdDate; private set => _createdDate = value; }
    public decimal TaxRate { get => _taxRate; set => _taxRate = (value >= 0 && value <= 1) ? value : throw new ArgumentException("Tax rate must be between 0 and 1"); }
    public bool IsTaxIncluded { get => _isTaxIncluded; set => _isTaxIncluded = value; }
    public string ReceiptNumber { get => _receiptNumber; set => _receiptNumber = value ?? string.Empty; }
    public string DepartmentCode { get => _departmentCode; set => _departmentCode = value ?? string.Empty; }

    // [NEW] Свойства новых атрибутов
    public int TransactionCount => _transactionCount;
    public decimal TotalProcessedAmount => _totalProcessedAmount;
    public DateTime LastTransactionDate => _lastTransactionDate;
    public bool IsFrozen => _isFrozen;
    public IReadOnlyList<string> TransactionHistory => _transactionHistory.AsReadOnly();

    public PaymentMethod()
    {
        PaymentMethodId = 1; MethodName = "Default"; MinAmount = 0; Currency = "USD";
        IsActive = true; Description = ""; CreatedDate = DateTime.Now;
        TaxRate = 0.20m; IsTaxIncluded = false; ReceiptNumber = "RCP-000000"; DepartmentCode = "DEP-01";
    }
    public PaymentMethod(int id, string name, decimal minAmount) : this() { PaymentMethodId = id; MethodName = name; MinAmount = minAmount; }
    public PaymentMethod(int id, string name, decimal minAmount, string currency, string description) : this(id, name, minAmount) { Currency = currency; Description = description; }

    protected void RecordTransaction(decimal amount, bool success, string message)
    {
        _transactionCount++;
        _lastTransactionDate = DateTime.Now;
        _transactionHistory.Add($"[{_lastTransactionDate:HH:mm:ss}] {amount} {Currency} - {(success ? "OK" : "FAIL")} - {message}");

        var args = new PaymentEventArgs(amount, success, message);
        if (success)
        {
            _totalProcessedAmount += amount;
            PaymentProcessed?.Invoke(this, args);
        }
        else
        {
            PaymentFailed?.Invoke(this, args);
        }
    }

    public string GetStatistics() =>
        $"Метод '{MethodName}': транзакций={_transactionCount}, сумма={_totalProcessedAmount} {Currency}, последняя={_lastTransactionDate:HH:mm:ss}, заморожен={_isFrozen}";

    public void Freeze() { _isFrozen = true; StatusChanged?.Invoke(this, EventArgs.Empty); Console.WriteLine($"[{MethodName}] Метод заморожен."); }
    public void Unfreeze() { _isFrozen = false; StatusChanged?.Invoke(this, EventArgs.Empty); Console.WriteLine($"[{MethodName}] Метод разморожен."); }
    public void ClearHistory() { _transactionHistory.Clear(); _transactionCount = 0; _totalProcessedAmount = 0; }

    public virtual bool ProcessPayment(decimal amount)
    {
        if (_isFrozen) { RecordTransaction(amount, false, "Метод заморожен"); Console.WriteLine($"[{MethodName}] Метод заморожен."); return false; }
        if (!IsActive) { RecordTransaction(amount, false, "Неактивен"); Console.WriteLine($"[{MethodName}] Payment method is inactive."); return false; }
        if (CustomAmountValidator != null && !CustomAmountValidator(amount))
        { RecordTransaction(amount, false, "Не прошёл CustomAmountValidator"); Console.WriteLine($"[{MethodName}] Custom validator rejected {amount}."); return false; }
        if (!CheckMinimumAmount(amount))
        { RecordTransaction(amount, false, "Ниже минимума"); Console.WriteLine($"[{MethodName}] Amount {amount} is below minimum."); return false; }

        Console.WriteLine($"[{MethodName}] Processed payment of {amount} {Currency}.");
        RecordTransaction(amount, true, "Успешно");
        return true;
    }

    public virtual bool ProcessPayment(decimal amount, string note) { Console.WriteLine($"[{MethodName}] Note: {note}"); return ProcessPayment(amount); }
    public virtual bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;
    public virtual bool CheckMinimumAmount(decimal amount, decimal fee) => (amount - fee) >= MinAmount;
    public virtual string GetPaymentDetails() => $"ID: {PaymentMethodId}, Name: {MethodName}, MinAmount: {MinAmount} {Currency}, Active: {IsActive}, TaxRate: {TaxRate:P0}, Транзакций: {_transactionCount}";
    public virtual string GetPaymentDetails(bool includeDescription) => includeDescription && !string.IsNullOrEmpty(Description) ? GetPaymentDetails() + $", Description: {Description}" : GetPaymentDetails();

    public void Activate() => IsActive = true;
    public void Deactivate() => IsActive = false;
    public void UpdateMinAmount(decimal newMin) => MinAmount = newMin;
    public decimal CalculateTax(decimal amount) => IsTaxIncluded ? amount - (amount / (1 + TaxRate)) : amount * TaxRate;
    public string PrintReceipt(decimal amount) => $"[Receipt #{ReceiptNumber}] Dept: {DepartmentCode} | Amount: {amount} {Currency} | Tax: {CalculateTax(amount):N2} {Currency}";
    public void SetTaxRate(decimal newRate, bool includeInPrice) { TaxRate = newRate; IsTaxIncluded = includeInPrice; }
    public bool ValidateTaxNumber(string taxNumber) => !string.IsNullOrWhiteSpace(taxNumber) && taxNumber.Length >= 10;
    public virtual bool Refund(decimal amount) { RecordTransaction(amount, true, "Возврат (base)"); Console.WriteLine($"[{MethodName}] Refund of {amount} {Currency} processed (default)."); return true; }

    void IAuditable.PerformAudit() => Console.WriteLine($"[Explicit Audit] Base PaymentMethod ID {PaymentMethodId} audited.");
    string IAuditable.GetAuditTrail() => $"Audit Trail for {MethodName}: Created at {CreatedDate}, Status: {IsActive}";

    public bool CanProcessWith(PaymentMethod other, decimal amount) => this.CheckMinimumAmount(amount) && other.CheckMinimumAmount(amount);
}

// ==========================================
// 3. ПРОИЗВОДНЫЕ КЛАССЫ
// ==========================================

// --- OnlinePayment ---
public class OnlinePayment : PaymentMethod, ISecurePayment, IAuditable
{
    private string _paymentUrl = "https://example.com/pay";
    private string _providerName = "DefaultProvider";
    private bool _isSecure = true;
    private string _securityToken = Guid.NewGuid().ToString();
    private string _sslCertificateHash = "SHA256_DEFAULT_HASH";
    private int _sessionTimeoutSeconds = 300;
    private bool _isTwoFactorEnabled = true;
    private string _gatewayIpAddress = "192.168.1.1";

    // [NEW]
    private int _failedAttempts;
    private string _lastLoginIp = "0.0.0.0";
    private int _requestsPerMinute;
    private bool _isFraudSuspected;

    public event EventHandler<string> FraudSuspected;

    public string PaymentUrl { get => _paymentUrl; set => _paymentUrl = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("URL cannot be empty"); }
    public string ProviderName { get => _providerName; set => _providerName = value ?? string.Empty; }
    public bool IsSecure { get => _isSecure; set => _isSecure = value; }
    public string SecurityToken { get => _securityToken; set => _securityToken = value ?? string.Empty; }
    public string SslCertificateHash { get => _sslCertificateHash; set => _sslCertificateHash = value; }
    public int SessionTimeoutSeconds { get => _sessionTimeoutSeconds; set => _sessionTimeoutSeconds = value; }
    public bool IsTwoFactorEnabled { get => _isTwoFactorEnabled; set => _isTwoFactorEnabled = value; }
    public string GatewayIpAddress { get => _gatewayIpAddress; set => _gatewayIpAddress = value; }

    public int FailedAttempts => _failedAttempts;
    public string LastLoginIp => _lastLoginIp;
    public int RequestsPerMinute => _requestsPerMinute;
    public bool IsFraudSuspected => _isFraudSuspected;

    public OnlinePayment() : base() { }
    public OnlinePayment(int id, string name, decimal minAmount, string url, string provider) : base(id, name, minAmount) { PaymentUrl = url; ProviderName = provider; }

    public override bool ProcessPayment(decimal amount)
    {
        if (_isFraudSuspected) { Console.WriteLine($"[{MethodName}] Отклонено: подозрение на мошенничество."); return false; }
        if (!base.ProcessPayment(amount)) return false;
        Console.WriteLine($"  -> Redirecting to {PaymentUrl} via {ProviderName}. Token: {SecurityToken}");
        return true;
    }

    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", URL: {PaymentUrl}, Provider: {ProviderName}, Secure: {IsSecure}, Fraud: {_isFraudSuspected}";
    public override bool Refund(decimal amount) { Console.WriteLine($"[{MethodName}] Online refund of {amount} {Currency}."); return true; }
    public string GetSecurityInfo() => $"Secure: {IsSecure}, Token: {SecurityToken}, IP: {GatewayIpAddress}";
    public string GeneratePaymentLink(decimal amount) => $"{PaymentUrl}?amount={amount}&currency={Currency}";
    public void RenewSecurityToken() { SecurityToken = Guid.NewGuid().ToString(); Console.WriteLine($"[{MethodName}] Token renewed."); }
    public bool VerifySslCertificate(string hash) => SslCertificateHash.Equals(hash, StringComparison.OrdinalIgnoreCase);
    public void ToggleTwoFactor(bool enable) => IsTwoFactorEnabled = enable;
    public string ExportGatewayConfig() => $"Gateway: {ProviderName}, IP: {GatewayIpAddress}, Timeout: {SessionTimeoutSeconds}s";

    public void RegisterFailedAttempt(string ipAddress)
    {
        _failedAttempts++;
        _lastLoginIp = ipAddress;
        if (_failedAttempts >= 3 && !_isFraudSuspected)
        {
            _isFraudSuspected = true;
            FraudSuspected?.Invoke(this, $"Превышено число попыток ({_failedAttempts}) с IP {ipAddress}");
        }
    }

    public void ResetFraudFlag() { _isFraudSuspected = false; _failedAttempts = 0; Console.WriteLine($"[{MethodName}] Fraud flag reset."); }
    public bool IsRateLimitExceeded(int limitPerMinute = 60) { _requestsPerMinute++; return _requestsPerMinute > limitPerMinute; }

    void IAuditable.PerformAudit() => Console.WriteLine($"[Explicit Audit] OnlinePayment '{ProviderName}' checked.");
    string IAuditable.GetAuditTrail() => $"Online Audit: URL={PaymentUrl}, 2FA={IsTwoFactorEnabled}";
}

// --- MobilePayment ---
public class MobilePayment : OnlinePayment
{
    private string _phoneNumber = "+1234567890";
    private string _operatorName = "DefaultOperator";
    private string _simCardId = "SIM-99999";
    private bool _isRoaming;
    private decimal _smsFee = 0.05m;
    private string _deviceOs = "Android";

    // [NEW]
    private int _smsNotificationsSent;
    private int _roamingDays;
    private double _dataUsageMb;
    private bool _isSmsEnabled = true;

    public event EventHandler<string> SmsSent;

    public string PhoneNumber { get => _phoneNumber; set => _phoneNumber = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Phone cannot be empty"); }
    public string OperatorName { get => _operatorName; set => _operatorName = value ?? string.Empty; }
    public string SimCardId { get => _simCardId; set => _simCardId = value; }
    public bool IsRoaming { get => _isRoaming; set => _isRoaming = value; }
    public decimal SmsFee { get => _smsFee; set => _smsFee = value; }
    public string DeviceOs { get => _deviceOs; set => _deviceOs = value; }

    public int SmsNotificationsSent => _smsNotificationsSent;
    public int RoamingDays => _roamingDays;
    public double DataUsageMb => _dataUsageMb;
    public bool IsSmsEnabled => _isSmsEnabled;

    public MobilePayment() : base() { }
    public MobilePayment(int id, string name, decimal minAmount, string url, string provider, string phone, string op) : base(id, name, minAmount, url, provider) { PhoneNumber = phone; OperatorName = op; }

    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount)) return false;
        Console.WriteLine($"  -> Mobile payment from {PhoneNumber} ({OperatorName}) confirmed.");
        return true;
    }
    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", Phone: {PhoneNumber}, Operator: {OperatorName}, Roaming: {IsRoaming}";
    public void ToggleRoaming(bool roam) => IsRoaming = roam;
    public decimal CalculateTotalWithSms(decimal amount) => amount + SmsFee;
    public void UpdateDeviceOs(string newOs) => DeviceOs = newOs;
    public string GetSimInfo() => $"SIM ID: {SimCardId}, Operator: {OperatorName}";

    public void SendSmsConfirmation(string message)
    {
        if (!_isSmsEnabled) { Console.WriteLine($"[{MethodName}] SMS отключены."); return; }
        _smsNotificationsSent++;
        Console.WriteLine($"[{MethodName}] SMS to {PhoneNumber}: {message}");
        SmsSent?.Invoke(this, message);
    }

    public void EnableSmsNotifications(bool enable) => _isSmsEnabled = enable;
    public void AddDataUsage(double mb) => _dataUsageMb += mb;
    public string GetMobileUsageReport() => $"Phone: {PhoneNumber}, SMS: {_smsNotificationsSent}, Data: {_dataUsageMb:F1} MB, Roaming days: {_roamingDays}";
}

// --- BankTransfer ---
public class BankTransfer : PaymentMethod, IAuditable
{
    // [NEW] Типизированный делегат для события LimitReached
    public delegate void LimitReachedHandler(BankTransfer sender, decimal amount);

    private string _bankData = "DefaultBankData";
    private decimal _commissionRate = 0.01m;
    private string _bankName = "DefaultBank";
    private string _accountNumber = "0000000000";
    private string _swiftCode = "DEFAULTSWIFT";
    private string _correspondentAccount = "30101810400000000225";
    private bool _isInstantTransfer;
    private decimal _dailyLimit = 50000m;

    // [NEW]
    private int _transfersToday;
    private decimal _totalToday;
    private bool _isBlocked;
    private readonly List<string> _auditLog = new List<string>();

    // [NEW] Типизированное событие
    public event LimitReachedHandler LimitReached;

    public string BankData { get => _bankData; set => _bankData = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Bank data cannot be empty"); }
    public decimal CommissionRate { get => _commissionRate; set => _commissionRate = (value >= 0 && value <= 1) ? value : throw new ArgumentException("Commission rate must be between 0 and 1"); }
    public string BankName { get => _bankName; set => _bankName = value ?? string.Empty; }
    public string AccountNumber { get => _accountNumber; set => _accountNumber = value ?? string.Empty; }
    public string SwiftCode { get => _swiftCode; set => _swiftCode = value ?? string.Empty; }
    public string CorrespondentAccount { get => _correspondentAccount; set => _correspondentAccount = value ?? string.Empty; }
    public bool IsInstantTransfer { get => _isInstantTransfer; set => _isInstantTransfer = value; }
    public decimal DailyLimit { get => _dailyLimit; set => _dailyLimit = value; }

    public int TransfersToday => _transfersToday;
    public decimal TotalToday => _totalToday;
    public bool IsBlocked => _isBlocked;

    public BankTransfer() : base() { }
    public BankTransfer(int id, string name, decimal minAmount, string bankData, decimal commissionRate, string bankName, string account) : base(id, name, minAmount)
    { BankData = bankData; CommissionRate = commissionRate; BankName = bankName; AccountNumber = account; }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount * CommissionRate;
        return (amount - commission) >= MinAmount;
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (_isBlocked) { Console.WriteLine($"[{MethodName}] Счёт заблокирован."); return false; }
        if (!IsActive) { Console.WriteLine($"[{MethodName}] Inactive."); return false; }
        if (amount > DailyLimit) { LimitReached?.Invoke(this, amount); Console.WriteLine($"[{MethodName}] Exceeds daily limit {DailyLimit}."); return false; }
        if (!CheckMinimumAmount(amount)) { Console.WriteLine($"[{MethodName}] Below minimum after commission."); return false; }
        if (!base.ProcessPayment(amount)) return false;

        decimal commission = amount * CommissionRate;
        _transfersToday++;
        _totalToday += amount;
        _auditLog.Add($"[{DateTime.Now:HH:mm:ss}] Transfer {amount} {Currency} (комиссия {commission})");
        Console.WriteLine($"  -> Commission: {commission}, Net: {amount - commission} {Currency}");
        return true;
    }

    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", Bank: {BankName}, SWIFT: {SwiftCode}, TransfersToday: {_transfersToday}";
    public decimal CalculateCommission(decimal amount) => amount * CommissionRate;
    public void UpdateDailyLimit(decimal newLimit) => DailyLimit = newLimit;
    public bool VerifySwiftCode(string swift) => SwiftCode.Equals(swift, StringComparison.OrdinalIgnoreCase);
    public void ToggleInstantTransfer(bool instant) => IsInstantTransfer = instant;
    public string GetBankRequisites() => $"Bank: {BankName}, Acc: {AccountNumber}, Corr: {CorrespondentAccount}, SWIFT: {SwiftCode}";

    public void RegisterTransfer(decimal amount)
    {
        _transfersToday++;
        _totalToday += amount;
        _auditLog.Add($"[{DateTime.Now:HH:mm:ss}] Manual transfer {amount} {Currency}");
    }

    public void BlockAccount(string reason) { _isBlocked = true; _auditLog.Add($"[{DateTime.Now:HH:mm:ss}] BLOCKED: {reason}"); Console.WriteLine($"[{MethodName}] Счёт заблокирован: {reason}"); }
    public void UnblockAccount() { _isBlocked = false; Console.WriteLine($"[{MethodName}] Счёт разблокирован."); }

    public string GetDailyReport() => $"Bank: {BankName}, TransfersToday: {_transfersToday}, TotalToday: {_totalToday} {Currency}, Blocked: {_isBlocked}";
    public IReadOnlyList<string> GetAuditLog() => _auditLog.AsReadOnly();

    void IAuditable.PerformAudit() => Console.WriteLine($"[Explicit Audit] BankTransfer SWIFT: {SwiftCode}");
    string IAuditable.GetAuditTrail() => $"Bank Audit: Bank={BankName}, DailyLimit={DailyLimit}";
}

// --- CashPayment ---
public class CashPayment : PaymentMethod, ILoggable
{
    private string _cashPickupPoint = "Main Office";
    private bool _isAvailable = true;
    private string _pickupAddress = "123 Main St";
    private string _workingHours = "9:00-18:00";
    private bool _isCashAccepted = true;
    private decimal _maxCashLimit = 10000m;
    private string _cashierName = "John Doe";
    private int _safeBoxNumber = 5;

    // [NEW]
    private int _clientsInQueue;
    private DateTime _shiftStartTime;
    private decimal _cashInDrawer;
    private int _receiptsIssued;

    public event EventHandler<DateTime> ShiftStarted;
    public event EventHandler<DateTime> ShiftEnded;

    public string CashPickupPoint { get => _cashPickupPoint; set => _cashPickupPoint = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Pickup point cannot be empty"); }
    public bool IsAvailable { get => _isAvailable; set => _isAvailable = value; }
    public string PickupAddress { get => _pickupAddress; set => _pickupAddress = value ?? string.Empty; }
    public string WorkingHours { get => _workingHours; set => _workingHours = value ?? string.Empty; }
    public bool IsCashAccepted { get => _isCashAccepted; set => _isCashAccepted = value; }
    public decimal MaxCashLimit { get => _maxCashLimit; set => _maxCashLimit = value; }
    public string CashierName { get => _cashierName; set => _cashierName = value ?? string.Empty; }
    public int SafeBoxNumber { get => _safeBoxNumber; set => _safeBoxNumber = value; }

    public int ClientsInQueue => _clientsInQueue;
    public DateTime ShiftStartTime => _shiftStartTime;
    public decimal CashInDrawer => _cashInDrawer;
    public int ReceiptsIssued => _receiptsIssued;

    public CashPayment() : base() { }
    public CashPayment(int id, string name, decimal minAmount, string pickupPoint, string address, string hours) : base(id, name, minAmount)
    { CashPickupPoint = pickupPoint; PickupAddress = address; WorkingHours = hours; }

    public override bool ProcessPayment(decimal amount)
    {
        if (!IsAvailable || !IsCashAccepted) { Console.WriteLine($"[{MethodName}] Cannot accept cash."); return false; }
        if (amount > MaxCashLimit) { Console.WriteLine($"[{MethodName}] Exceeds max cash limit {MaxCashLimit}."); return false; }
        if (!base.ProcessPayment(amount)) return false;
        _cashInDrawer += amount;
        _receiptsIssued++;
        return true;
    }

    public override bool Refund(decimal amount) { _cashInDrawer -= amount; Console.WriteLine($"[{MethodName}] Cash refund {amount}."); return true; }
    public override string GetPaymentDetails() => base.GetPaymentDetails() + $", Pickup: {CashPickupPoint}, Cashier: {CashierName}, CashInDrawer: {_cashInDrawer}";
    public void Log(string message) => Console.WriteLine($"[LOG][{MethodName}] {message}");
    public void AssignCashier(string name) => CashierName = name;
    public void UpdateMaxLimit(decimal limit) => MaxCashLimit = limit;
    public void ToggleCashAcceptance(bool accept) => IsCashAccepted = accept;
    public string GetSafeStatus() => $"Safe #{SafeBoxNumber} at {CashPickupPoint}, Cashier: {CashierName}";

    public void EnqueueClient() => _clientsInQueue++;
    public void ServeNextClient()
    {
        if (_clientsInQueue > 0) { _clientsInQueue--; Console.WriteLine($"[{MethodName}] Обслужен клиент. В очереди: {_clientsInQueue}"); }
        else Console.WriteLine($"[{MethodName}] Очередь пуста.");
    }

    public void StartShift() { _shiftStartTime = DateTime.Now; ShiftStarted?.Invoke(this, _shiftStartTime); Console.WriteLine($"[{MethodName}] Смена начата."); }
    public void EndShift() { ShiftEnded?.Invoke(this, DateTime.Now); Console.WriteLine($"[{MethodName}] Смена завершена. Касса: {_cashInDrawer}, чеков: {_receiptsIssued}"); }
    public void AddCashToDrawer(decimal amount) => _cashInDrawer += amount;
}

// ==========================================
// 4. DI-ПРОЦЕССОР
// ==========================================
public class PaymentProcessor
{
    private readonly ILoggable _logger;
    private readonly IRefundable _refundablePayment;

    public Action<bool, decimal> OnRefundCompleted { get; set; }

    public PaymentProcessor(ILoggable logger, IRefundable refundablePayment)
    {
        _logger = logger ?? throw new ArgumentNullException(nameof(logger));
        _refundablePayment = refundablePayment ?? throw new ArgumentNullException(nameof(refundablePayment));
    }

    public void ExecuteRefundFlow(decimal amount)
    {
        _logger.Log($"Starting refund for {amount}");
        bool success = _refundablePayment.Refund(amount);
        _logger.Log(success ? "Refund OK." : "Refund FAILED.");
        OnRefundCompleted?.Invoke(success, amount);
    }
}

// ==========================================
// 5. PaymentManager — коллекции + делегаты + события
// ==========================================
public class PaymentManager
{
    private readonly List<PaymentMethod> _allMethods = new List<PaymentMethod>();
    private readonly Dictionary<int, PaymentMethod> _byId = new Dictionary<int, PaymentMethod>();
    private readonly HashSet<string> _processedReceipts = new HashSet<string>();

    // [ИСПРАВЛЕНО] Типизированный делегат — sender типа PaymentMethod
    public event PaymentNotificationHandler AnyPaymentProcessed;
    public event PaymentNotificationHandler AnyPaymentFailed;
    public event EventHandler<string> MethodRegistered;

    public Func<PaymentMethod, bool> RegistrationFilter { get; set; }

    public IReadOnlyList<PaymentMethod> AllMethods => _allMethods.AsReadOnly();

    public bool Register(PaymentMethod method)
    {
        if (method == null) throw new ArgumentNullException(nameof(method));
        if (RegistrationFilter != null && !RegistrationFilter(method))
        {
            Console.WriteLine($"[Manager] '{method.MethodName}' отклонён фильтром.");
            return false;
        }
        if (_byId.ContainsKey(method.PaymentMethodId)) { Console.WriteLine($"[Manager] ID {method.PaymentMethodId} уже зарегистрирован."); return false; }

        _allMethods.Add(method);
        _byId[method.PaymentMethodId] = method;

        method.PaymentProcessed += (s, e) => AnyPaymentProcessed?.Invoke(s, e);
        method.PaymentFailed   += (s, e) => AnyPaymentFailed?.Invoke(s, e);

        MethodRegistered?.Invoke(this, method.MethodName);
        return true;
    }

    public PaymentMethod FindById(int id) => _byId.TryGetValue(id, out var m) ? m : null;

    public IEnumerable<PaymentMethod> Filter(Func<PaymentMethod, bool> predicate) => _allMethods.Where(predicate);

    public decimal TotalProcessed => _allMethods.Sum(m => m.TotalProcessedAmount);
    public int TotalTransactions => _allMethods.Sum(m => m.TransactionCount);

    public bool ProcessAndTrack(PaymentMethod method, decimal amount)
    {
        bool ok = method.ProcessPayment(amount);
        if (ok) _processedReceipts.Add($"{method.PaymentMethodId}-{amount}-{DateTime.Now.Ticks}");
        return ok;
    }

    public string GetSummaryReport()
    {
        var sb = new System.Text.StringBuilder();
        sb.AppendLine("=== Сводный отчёт ===");
        foreach (var m in _allMethods) sb.AppendLine(m.GetStatistics());
        sb.AppendLine($"ИТОГО: {TotalTransactions} транзакций, сумма {TotalProcessed}");
        return sb.ToString();
    }
}

// ==========================================
// 6. ДЕМОНСТРАЦИЯ
// ==========================================
Console.WriteLine("=== 1. Создание и регистрация в PaymentManager ===");
var manager = new PaymentManager();
manager.MethodRegistered += (s, name) => Console.WriteLine($"[EVENT] Зарегистрирован метод: {name}");

// sender типизирован как PaymentMethod — s.MethodName доступен без каста
manager.AnyPaymentProcessed += (s, e) => Console.WriteLine($"[EVENT] ✓ Платёж {e.Amount} через {s.MethodName}");
manager.AnyPaymentFailed    += (s, e) => Console.WriteLine($"[EVENT] ✗ Ошибка в {s.MethodName}: {e.Message}");

var baseP  = new PaymentMethod(1, "Base Card", 10m);
var online = new OnlinePayment(2, "Online", 20m, "https://pay.example.com", "SecurePay");
var mobile = new MobilePayment(3, "Mobile", 5m, "https://m.example.com", "MobPay", "+79001234567", "MTS");
var bank   = new BankTransfer(4, "Bank", 100m, "BD", 0.01m, "Sber", "40817810000000000001");
var cash   = new CashPayment(5, "Cash", 1m, "Office #1", "Lenin 10", "10:00-20:00");

manager.Register(baseP); manager.Register(online); manager.Register(mobile); manager.Register(bank); manager.Register(cash);
Console.WriteLine();

Console.WriteLine("=== 2. Подписка на специфичные события ===");
online.FraudSuspected += (s, msg) => Console.WriteLine($"[FRAUD] {msg}");
mobile.SmsSent        += (s, msg) => Console.WriteLine($"[SMS-SENT] {msg}");

// sender типизирован как BankTransfer — s.DailyLimit доступен без каста
bank.LimitReached += (s, amount) => Console.WriteLine($"[LIMIT] Попытка перевода {amount}, лимит {s.DailyLimit}");

cash.ShiftStarted += (s, t) => Console.WriteLine($"[SHIFT] Начало смены: {t:HH:mm:ss}");
cash.ShiftEnded   += (s, t) => Console.WriteLine($"[SHIFT] Конец смены: {t:HH:mm:ss}");
Console.WriteLine();

Console.WriteLine("=== 3. Использование делегата CustomAmountValidator ===");
baseP.CustomAmountValidator = amount => amount % 10 == 0;
Console.WriteLine("-- 15 (не пройдёт):"); manager.ProcessAndTrack(baseP, 15);
Console.WriteLine("-- 20 (пройдёт):");    manager.ProcessAndTrack(baseP, 20);
Console.WriteLine();

Console.WriteLine("=== 4. Транзакции через manager ===");
manager.ProcessAndTrack(online, 100);
manager.ProcessAndTrack(mobile, 50);
manager.ProcessAndTrack(bank, 500);
Console.WriteLine("-- Превышение лимита банка:");
manager.ProcessAndTrack(bank, 60000);
Console.WriteLine();

Console.WriteLine("=== 5. Rate-limit и fraud для OnlinePayment ===");
online.RegisterFailedAttempt("10.0.0.1");
online.RegisterFailedAttempt("10.0.0.2");
online.RegisterFailedAttempt("10.0.0.3");
manager.ProcessAndTrack(online, 100);
online.ResetFraudFlag();
manager.ProcessAndTrack(online, 100);
Console.WriteLine();

Console.WriteLine("=== 6. Смена, очередь и SMS ===");
cash.StartShift();
cash.EnqueueClient(); cash.EnqueueClient();
cash.ServeNextClient();
cash.AddCashToDrawer(500);
mobile.SendSmsConfirmation("Платёж подтверждён");
cash.EndShift();
Console.WriteLine();

Console.WriteLine("=== 7. Коллекции и фильтры ===");
Console.WriteLine("-- Только онлайн-методы:");
foreach (var m in manager.Filter(x => x is OnlinePayment)) Console.WriteLine($"  {m.MethodName}");
Console.WriteLine($"-- Поиск по ID=4: {manager.FindById(4)?.MethodName}");
Console.WriteLine($"-- Всего обработано: {manager.TotalProcessed}, транзакций: {manager.TotalTransactions}");
Console.WriteLine();

Console.WriteLine(manager.GetSummaryReport());

=== 1. Создание и регистрация в PaymentManager ===
[EVENT] Зарегистрирован метод: Base Card
[EVENT] Зарегистрирован метод: Online
[EVENT] Зарегистрирован метод: Mobile
[EVENT] Зарегистрирован метод: Bank
[EVENT] Зарегистрирован метод: Cash

=== 2. Подписка на специфичные события ===

=== 3. Использование делегата CustomAmountValidator ===
-- 15 (не пройдёт):
[EVENT] ✗ Ошибка в Base Card: Не прошёл CustomAmountValidator
[Base Card] Custom validator rejected 15.
-- 20 (пройдёт):
[Base Card] Processed payment of 20 USD.
[EVENT] ✓ Платёж 20 через Base Card

=== 4. Транзакции через manager ===
[Online] Processed payment of 100 USD.
[EVENT] ✓ Платёж 100 через Online
  -> Redirecting to https://pay.example.com via SecurePay. Token: ebf886fa-29e6-4bdf-815d-482d2a6238fa
[Mobile] Processed payment of 50 USD.
[EVENT] ✓ Платёж 50 через Mobile
  -> Redirecting to https://m.example.com via MobPay. Token: 8f1e83b6-0d08-4847-ac97-535690f36070
  -> Mobile payment from +79001234567 (MTS) confirmed.
[Ban